In [10]:
import pandas as pd
import rapidfuzz

In [9]:
!pip install rapidfuzz

python3.11(73839) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 1.3 MB/s  0:00:01 eta 0:00:01


In [7]:
tracks = pd.read_csv("./data/archive/tracks.csv")

In [12]:
import ast
import re
import unicodedata
from collections import defaultdict
from dataclasses import dataclass

import numpy as np
import pandas as pd
from rapidfuzz import fuzz


# ============================================================
# CONFIG
# ============================================================

FEATURE_COLS = [
    "danceability",
    "energy",
    "valence",
    "tempo",
    "loudness",
    "acousticness",
    "speechiness",
]

# soglie conservative: duplicate "forti"
TITLE_SIM_THRESHOLD = 88          # 0..100
FEATURE_DIST_THRESHOLD = 1.35     # distanza euclidea dopo scaling
DURATION_DIFF_MAX_SEC = 12
YEAR_DIFF_MAX = 20

# blocchi candidati
BLOCK_TITLE_WORDS = 3             # prime 3 parole del titolo normalizzato
MAX_BLOCK_SIZE = 80               # per evitare confronti enormi


# ============================================================
# HELPERS
# ============================================================

def parse_listish(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, float) and pd.isna(x):
        return []
    s = str(x).strip()
    if not s:
        return []
    try:
        out = ast.literal_eval(s)
        if isinstance(out, list):
            return [str(v).strip() for v in out if str(v).strip()]
    except Exception:
        pass
    return [s]


def strip_accents(s: str) -> str:
    s = unicodedata.normalize("NFKD", s)
    return "".join(ch for ch in s if not unicodedata.combining(ch))


def normalize_text(s: str) -> str:
    s = str(s or "").lower()
    s = strip_accents(s)

    # rimuovi contenuto tra parentesi
    s = re.sub(r"\([^)]*\)", " ", s)
    s = re.sub(r"\[[^\]]*\]", " ", s)

    # taglia tutto dopo trattini con parole "versionose"
    # es: "Pensa - Sanremo 2007", "Song - Remastered 2011"
    s = re.sub(
        r"\s*-\s*(live|remaster(?:ed)?|version|acoustic|karaoke|instrumental|sanremo|radio edit|edit|deluxe).*$",
        "",
        s,
        flags=re.IGNORECASE,
    )

    # rimuovi feat / featuring dal titolo
    s = re.sub(r"\b(feat|ft|featuring)\b.*$", "", s, flags=re.IGNORECASE)

    # tieni lettere/numeri/spazi/apostrofi
    s = re.sub(r"[^a-z0-9\s']", " ", s)

    # rimuovi anni isolati
    s = re.sub(r"\b(19|20)\d{2}\b", " ", s)

    # compatta spazi
    s = re.sub(r"\s+", " ", s).strip()

    return s


def main_artist(artists_value):
    artists = parse_listish(artists_value)
    if not artists:
        return ""
    return normalize_text(artists[0])


def extract_year(x):
    s = str(x or "").strip()
    if not s:
        return np.nan
    m = re.match(r"^(\d{4})", s)
    if not m:
        return np.nan
    return int(m.group(1))


def title_block_key(title_norm: str, n_words: int = BLOCK_TITLE_WORDS) -> str:
    words = title_norm.split()
    if not words:
        return ""
    return " ".join(words[:n_words])


class UnionFind:
    def __init__(self, items):
        self.parent = {x: x for x in items}
        self.rank = {x: 0 for x in items}

    def find(self, x):
        p = self.parent[x]
        if p != x:
            self.parent[x] = self.find(p)
        return self.parent[x]

    def union(self, a, b):
        ra = self.find(a)
        rb = self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            self.parent[ra] = rb
        elif self.rank[ra] > self.rank[rb]:
            self.parent[rb] = ra
        else:
            self.parent[rb] = ra
            self.rank[ra] += 1


# ============================================================
# PREP
# ============================================================

def prepare_tracks(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # standardizza tipi
    df["name"] = df["name"].astype(str)
    df["main_artist"] = df["artists"].apply(main_artist)
    df["title_norm"] = df["name"].apply(normalize_text)
    df["title_block"] = df["title_norm"].apply(title_block_key)
    df["release_year"] = df["release_date"].apply(extract_year)

    # durata in secondi
    df["duration_sec"] = (df["duration_ms"].astype(float) / 1000.0).round(1)

    # filtra righe inutili
    df = df[df["main_artist"] != ""].copy()
    df = df[df["title_norm"] != ""].copy()

    # feature numeriche
    for col in FEATURE_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # scala grossolana delle feature per rendere la distanza sensata
    # niente StandardScaler esterno: qui stiamo leggeri e portabili
    df["tempo_scaled"] = df["tempo"] / 200.0
    df["loudness_scaled"] = (df["loudness"] + 60.0) / 60.0

    # matrice features finale
    df["f_danceability"] = df["danceability"]
    df["f_energy"] = df["energy"]
    df["f_valence"] = df["valence"]
    df["f_tempo"] = df["tempo_scaled"]
    df["f_loudness"] = df["loudness_scaled"]
    df["f_acousticness"] = df["acousticness"]
    df["f_speechiness"] = df["speechiness"]

    return df


# ============================================================
# DUPLICATE SCORE
# ============================================================

@dataclass
class PairScore:
    is_duplicate: bool
    title_sim: float
    feat_dist: float
    duration_diff: float
    year_diff: float


def compute_feature_distance(row_a, row_b) -> float:
    va = np.array([
        row_a["f_danceability"],
        row_a["f_energy"],
        row_a["f_valence"],
        row_a["f_tempo"],
        row_a["f_loudness"],
        row_a["f_acousticness"],
        row_a["f_speechiness"],
    ], dtype=float)

    vb = np.array([
        row_b["f_danceability"],
        row_b["f_energy"],
        row_b["f_valence"],
        row_b["f_tempo"],
        row_b["f_loudness"],
        row_b["f_acousticness"],
        row_b["f_speechiness"],
    ], dtype=float)

    if np.isnan(va).any() or np.isnan(vb).any():
        return np.inf

    return float(np.linalg.norm(va - vb))


def compare_rows(row_a, row_b) -> PairScore:
    # similarità titolo
    title_sim = fuzz.ratio(row_a["title_norm"], row_b["title_norm"])

    # feature distance
    feat_dist = compute_feature_distance(row_a, row_b)

    # durata
    duration_diff = abs(float(row_a["duration_sec"]) - float(row_b["duration_sec"]))

    # anno
    year_a = row_a["release_year"]
    year_b = row_b["release_year"]
    if pd.isna(year_a) or pd.isna(year_b):
        year_diff = 0
    else:
        year_diff = abs(float(year_a) - float(year_b))

    # regole duplicate
    # approccio robusto:
    # - titolo molto simile
    # - feature vicine
    # - durata simile
    # - anno non troppo distante
    # Nota: anno molto lontano è possibile con ripubblicazioni,
    # quindi non lo usiamo come veto assoluto.
    is_duplicate = (
        title_sim >= TITLE_SIM_THRESHOLD
        and feat_dist <= FEATURE_DIST_THRESHOLD
        and duration_diff <= DURATION_DIFF_MAX_SEC
        and year_diff <= YEAR_DIFF_MAX
    )

    return PairScore(
        is_duplicate=is_duplicate,
        title_sim=title_sim,
        feat_dist=feat_dist,
        duration_diff=duration_diff,
        year_diff=year_diff,
    )


# ============================================================
# CLUSTERING BY BLOCKS
# ============================================================

def find_duplicate_pairs(df: pd.DataFrame) -> pd.DataFrame:
    """
    Restituisce un dataframe con le coppie duplicate stimate.
    Funziona a blocchi: main_artist + title_block
    """
    rows = []
    grouped = df.groupby(["main_artist", "title_block"], sort=False)

    for (artist, block), g in grouped:
        if len(g) < 2:
            continue

        # blocchi troppo grandi = pericolo combinatorio
        # in quel caso prendiamo solo titoli quasi identici
        g = g.sort_values(["popularity", "name"], ascending=[False, True]).copy()

        # per 500k righe è importante contenere i confronti
        if len(g) > MAX_BLOCK_SIZE:
            g = g[g["title_norm"].map(len) > 0].copy()
            # ulteriore restrizione: dentro blocchi grossi, spezza per primo token + durata bucket
            g["dur_bucket"] = (g["duration_sec"] // 10).astype("Int64")
            subgroups = g.groupby(["title_block", "dur_bucket"], sort=False)
        else:
            subgroups = [(None, g)]

        for _, subg in subgroups:
            if len(subg) < 2:
                continue

            recs = subg.to_dict("records")
            n = len(recs)

            for i in range(n):
                for j in range(i + 1, n):
                    a = recs[i]
                    b = recs[j]

                    score = compare_rows(a, b)
                    if score.is_duplicate:
                        rows.append({
                            "id_a": a["id"],
                            "id_b": b["id"],
                            "name_a": a["name"],
                            "name_b": b["name"],
                            "artist": a["main_artist"],
                            "title_sim": score.title_sim,
                            "feat_dist": score.feat_dist,
                            "duration_diff": score.duration_diff,
                            "year_diff": score.year_diff,
                        })

    return pd.DataFrame(rows)


def assign_duplicate_clusters(df: pd.DataFrame, pairs_df: pd.DataFrame) -> pd.DataFrame:
    """
    Usa union-find per creare cluster di duplicati.
    """
    df = df.copy()
    uf = UnionFind(df["id"].tolist())

    if not pairs_df.empty:
        for row in pairs_df.itertuples(index=False):
            uf.union(row.id_a, row.id_b)

    df["dup_cluster"] = df["id"].map(uf.find)
    return df


def choose_cluster_representative(df: pd.DataFrame) -> pd.DataFrame:
    """
    Tiene il brano migliore di ogni cluster.
    Ordinamento consigliato:
    1) popularity desc
    2) explicit asc
    3) release_year desc
    """
    df = df.copy()

    # explicit: preferisci non explicit a parità, se vuoi
    df["explicit_sort"] = pd.to_numeric(df["explicit"], errors="coerce").fillna(0)

    # preferisci il più popolare, poi non-explicit, poi più recente
    df = df.sort_values(
        by=["dup_cluster", "popularity", "explicit_sort", "release_year"],
        ascending=[True, False, True, False]
    )

    dedup = df.drop_duplicates(subset=["dup_cluster"], keep="first").copy()
    return dedup


# ============================================================
# MAIN PIPELINE
# ============================================================

def deduplicate_tracks(df: pd.DataFrame):
    print(f"[INFO] input rows: {len(df):,}")

    prepared = prepare_tracks(df)
    print(f"[INFO] after prepare: {len(prepared):,}")

    pairs = find_duplicate_pairs(prepared)
    print(f"[INFO] duplicate pairs found: {len(pairs):,}")

    clustered = assign_duplicate_clusters(prepared, pairs)

    # dimensione cluster
    cluster_sizes = clustered.groupby("dup_cluster")["id"].transform("size")
    clustered["dup_cluster_size"] = cluster_sizes

    dedup = choose_cluster_representative(clustered)
    print(f"[INFO] deduplicated rows: {len(dedup):,}")
    print(f"[INFO] removed rows: {len(clustered) - len(dedup):,}")

    return prepared, pairs, clustered, dedup


# ============================================================
# EXAMPLE USAGE
# ============================================================

if __name__ == "__main__":
    # Scegli uno dei due:
    # df = pd.read_csv("tracks_test_500k.csv")
    df = tracks

    prepared, pairs, clustered, dedup = deduplicate_tracks(df)

    # salva output utili
    pairs.to_csv("duplicate_pairs.csv", index=False)
    clustered.to_parquet("tracks_with_duplicate_clusters.parquet", index=False)
    dedup.to_parquet("tracks_deduplicated.parquet", index=False)

    # debug: cluster sospetti
    suspicious = (
        clustered[clustered["dup_cluster_size"] > 1]
        .sort_values(["dup_cluster", "popularity"], ascending=[True, False])
        [["dup_cluster", "id", "name", "artists", "popularity", "release_date",
          "duration_ms", "danceability", "energy", "valence", "tempo", "loudness"]]
    )
    suspicious.to_csv("duplicate_clusters_debug.csv", index=False)

    print("[INFO] done.")

[INFO] input rows: 586,672
[INFO] after prepare: 535,409
[INFO] duplicate pairs found: 244,900
[INFO] deduplicated rows: 463,370
[INFO] removed rows: 72,039
[INFO] done.
